# 05 — SAR Corridor Analysis

> **CONTEXTUAL REMOTE-SENSING PRODUCT NOTICE**
> All SAR layers in this notebook are derived from Sentinel-1 / PALSAR-2
> backscatter data. They do **not** represent exact electrical asset geometry
> and **must not** be used for engineering or operational decisions without
> utility-authoritative verification.
> `sar_confidence` is kept independent of grid topology `confidence` throughout.

This notebook:
1. Extracts SAR corridor footprints by buffering IESO transmission lines
2. Visualises corridor polygons over the pilot area
3. Summarises per-corridor VV backscatter statistics (mean/std/p10/p90 dB)
4. Runs bi-temporal change detection across corridors

Buffer widths: 500 kV → 120 m | 230 kV → 60 m | 115 kV → 40 m | distribution → 15 m

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from shapely.geometry import box

from src.sar import check_sar_enabled, _SAR_DISCLAIMER
from src.sar.corridor_extractor import extract_corridor_footprints
from src.sar.change_detector import compute_backscatter_change
from src.ingestion.ieso_fetcher import IESOFetcher
from src.utils.config_loader import load_settings, load_sar_settings

cfg = load_settings()
sar_cfg = load_sar_settings().get("sar", {})

# SAR disclaimer — printed at the top of every cell that displays SAR-derived results
print("=" * 70)
print("SAR DISCLAIMER:")
print(_SAR_DISCLAIMER)
print("=" * 70)
print()

sar_enabled = check_sar_enabled()
print(f"SAR module enabled: {sar_enabled}")

bbox_cfg = cfg["region"]["bbox"]
BBOX = (bbox_cfg["west"], bbox_cfg["south"], bbox_cfg["east"], bbox_cfg["north"])

## 5.1  Load IESO Transmission Lines

In [ ]:
# SAR DISCLAIMER: outputs from this section are SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

ieso = IESOFetcher()
tx_registry = ieso.fetch_transmission_registry()

# Filter to line geometries only
tx_lines = tx_registry[
    tx_registry.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

bbox_poly = box(*BBOX)
tx_lines_clip = tx_lines[tx_lines.intersects(bbox_poly)].copy()

print(f"IESO TX lines (province-wide): {len(tx_lines):,}")
print(f"IESO TX lines in bbox:         {len(tx_lines_clip):,}")
print()
print("By voltage tier:")
print(tx_lines_clip["voltage_kv"].value_counts().to_string())

## 5.2  Extract SAR Corridor Footprints

In [ ]:
# SAR DISCLAIMER: corridor footprints are contextual RS products
print(_SAR_DISCLAIMER)
print()

SAR_VV_PATH = Path(cfg["sources"]["sar"]["sentinel1_vv_mosaic"])

if not SAR_VV_PATH.exists():
    print(f"SAR mosaic not found at {SAR_VV_PATH}")
    print("Running corridor extraction without raster statistics.")
    SAR_VV_PATH = None
else:
    print(f"SAR mosaic: {SAR_VV_PATH}")

buffer_widths_m = {500: 120, 230: 60, 115: 40, 0: 15}

corridors = extract_corridor_footprints(
    transmission_lines=tx_lines_clip,
    buffer_widths_m=buffer_widths_m,
    sar_mosaic_path=SAR_VV_PATH,
)

print(f"Corridors extracted: {len(corridors):,}")
print(f"CRS: {corridors.crs}")
print()

# Verify mandatory SAR metadata tags are present
required_cols = ["source", "is_exact_asset_geometry", "disclaimer_text", "sar_confidence"]
print("Mandatory SAR metadata check:")
for col in required_cols:
    val = corridors[col].iloc[0] if col in corridors.columns and len(corridors) > 0 else "MISSING"
    status = "OK" if col in corridors.columns else "MISSING"
    print(f"  {col:<30} {status}  (sample: {str(val)[:60]})")

print()
print("sar_confidence distribution:")
if "sar_confidence" in corridors.columns:
    print(corridors["sar_confidence"].value_counts(dropna=False).to_string())

## 5.3  Corridor Footprint Map

In [ ]:
# SAR DISCLAIMER: corridor polygons are contextual RS products
print(_SAR_DISCLAIMER)
print()

fig, ax = plt.subplots(figsize=(12, 9))
ax.set_facecolor("#1a1a2e")

voltage_color_map = {
    500: "#e74c3c",
    230: "#e67e22",
    115: "#f1c40f",
    0:   "#2ecc71",
}

def voltage_to_color(v):
    if v >= 450: return voltage_color_map[500]
    if v >= 200: return voltage_color_map[230]
    if v >= 100: return voltage_color_map[115]
    return voltage_color_map[0]

for _, row in corridors.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    color = voltage_to_color(float(row.get("voltage_kv", 0) or 0))
    try:
        x, y = geom.exterior.xy
        ax.fill(x, y, alpha=0.30, color=color)
        ax.plot(x, y, color=color, linewidth=0.6, alpha=0.7)
    except Exception:
        # MultiPolygon fallback
        for part in geom.geoms:
            xp, yp = part.exterior.xy
            ax.fill(xp, yp, alpha=0.30, color=color)

# Overlay TX centrelines
if len(tx_lines_clip) > 0:
    tx_lines_clip.plot(ax=ax, linewidth=1.2, color="white", alpha=0.5)

legend_handles = [
    mpatches.Patch(color=voltage_color_map[500], alpha=0.6, label="500 kV (120 m buffer)"),
    mpatches.Patch(color=voltage_color_map[230], alpha=0.6, label="230 kV (60 m buffer)"),
    mpatches.Patch(color=voltage_color_map[115], alpha=0.6, label="115 kV (40 m buffer)"),
    mpatches.Patch(color=voltage_color_map[0],   alpha=0.6, label="Distribution (15 m buffer)"),
    mlines.Line2D([0], [0], color="white", linewidth=1.2, alpha=0.5, label="TX centreline"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right",
          facecolor="#2c3e50", labelcolor="white")

ax.set_xlim(BBOX[0], BBOX[2])
ax.set_ylim(BBOX[1], BBOX[3])
ax.set_xlabel("Longitude", color="white")
ax.set_ylabel("Latitude", color="white")
ax.tick_params(colors="white")
ax.set_title(
    f"SAR Corridor Footprints over TX Lines\n"
    f"{len(corridors):,} corridors | source=SAR_contextual_RS | is_exact_asset_geometry=False",
    fontsize=11, fontweight="bold", color="white"
)

plt.tight_layout()
plt.savefig("../data/outputs/05_sar_corridor_footprints.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.4  Per-Corridor VV Backscatter Statistics

In [ ]:
# SAR DISCLAIMER: backscatter statistics are SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

stats_cols = ["corridor_id", "voltage_kv", "buffer_width_m", "length_km",
              "mean_vv_db", "std_vv_db", "p10_vv_db", "p90_vv_db", "sar_confidence"]
available_cols = [c for c in stats_cols if c in corridors.columns]
stats_df = corridors[available_cols].copy()

has_sar_data = "mean_vv_db" in stats_df.columns and stats_df["mean_vv_db"].notna().any()

if has_sar_data:
    stats_with_data = stats_df[stats_df["mean_vv_db"].notna()].copy()
    print(f"Corridors with SAR statistics: {len(stats_with_data):,} / {len(stats_df):,}")
    print()
    print("VV backscatter summary by voltage tier (dB):")
    print(
        stats_with_data
        .groupby("voltage_kv")[["mean_vv_db", "std_vv_db", "p10_vv_db", "p90_vv_db"]]
        .agg(["mean", "min", "max"])
        .round(2)
        .to_string()
    )
else:
    print("SAR raster not available — generating synthetic stats for visualisation demo.")
    print("[sar_confidence will be set to 'unvalidated' for all corridors]")
    print()
    rng = np.random.default_rng(seed=7)
    n = len(corridors)
    # Simulate realistic VV dB values: forested/vegetated corridors ~-12 dB, open ~-8 dB
    base_vv = np.where(stats_df["voltage_kv"] >= 230, -10.2, -12.5) if "voltage_kv" in stats_df else -12.0
    stats_df["mean_vv_db"] = base_vv + rng.normal(0, 1.8, n)
    stats_df["std_vv_db"]  = rng.uniform(2.0, 5.5, n)
    stats_df["p10_vv_db"]  = stats_df["mean_vv_db"] - stats_df["std_vv_db"] * 1.28
    stats_df["p90_vv_db"]  = stats_df["mean_vv_db"] + stats_df["std_vv_db"] * 1.28
    stats_df["sar_confidence"] = "unvalidated"
    print(stats_df[["corridor_id", "voltage_kv", "mean_vv_db", "std_vv_db",
                    "sar_confidence"]].head(12).to_string(index=False))

In [ ]:
# SAR DISCLAIMER: visualisation of contextual RS products
print(_SAR_DISCLAIMER)
print()

n_unvalidated = (stats_df["sar_confidence"] == "unvalidated").sum()
n_uncertain   = (stats_df["sar_confidence"] == "uncertain").sum()
print(f"sar_confidence='unvalidated': {n_unvalidated:,} corridors")
print(f"sar_confidence='uncertain':   {n_uncertain:,} corridors  (spring-melt artefact flag)")
if n_uncertain > 0:
    print()
    print("WARNING: corridors with sar_confidence='uncertain' carry a spring-melt artefact")
    print("         flag. Results are less reliable — consult notebook 06 for suppression demo.")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "SAR Corridor VV Backscatter Statistics\n"
    "[source=SAR_contextual_RS | is_exact_asset_geometry=False]",
    fontsize=12, fontweight="bold"
)

# Distribution of mean VV dB by voltage tier
ax = axes[0]
voltage_groups = {
    "500 kV":   stats_df[stats_df["voltage_kv"] >= 450]["mean_vv_db"],
    "230 kV":   stats_df[(stats_df["voltage_kv"] >= 200) & (stats_df["voltage_kv"] < 450)]["mean_vv_db"],
    "115 kV":   stats_df[(stats_df["voltage_kv"] >= 100) & (stats_df["voltage_kv"] < 200)]["mean_vv_db"],
    "Dist.":    stats_df[stats_df["voltage_kv"] < 100]["mean_vv_db"],
}
tier_colors = ["#c0392b", "#e67e22", "#f1c40f", "#2ecc71"]
for (label, data), color in zip(voltage_groups.items(), tier_colors):
    if len(data.dropna()) > 0:
        ax.hist(data.dropna(), bins=15, alpha=0.65, color=color, label=label, edgecolor="white")
ax.axvline(-15, color="#2c3e50", linestyle="--", linewidth=1.2, label="Dry land ref. (~-15 dB)")
ax.set_xlabel("Mean VV (dB)")
ax.set_ylabel("Corridor count")
ax.set_title("Mean VV Distribution by Voltage Tier")
ax.legend(fontsize=9)

# Error bar plot: mean ± std per corridor (top 20 by voltage)
ax = axes[1]
top_n = stats_df.sort_values("voltage_kv", ascending=False).head(20)
top_n = top_n.dropna(subset=["mean_vv_db", "std_vv_db"])
if len(top_n) > 0:
    y = range(len(top_n))
    ax.barh(list(y), top_n["mean_vv_db"].values, xerr=top_n["std_vv_db"].values,
            color="#3498db", alpha=0.75, edgecolor="white", capsize=3)
    ax.set_yticks(list(y))
    ax.set_yticklabels(
        [f"{row['corridor_id']} ({row.get('voltage_kv', 0):.0f} kV)"
         for _, row in top_n.iterrows()],
        fontsize=7
    )
    ax.axvline(-15, color="#e74c3c", linestyle="--", linewidth=1.2)
    ax.set_xlabel("Mean VV (dB) \u00b1 1\u03c3")
    ax.set_title("Top-20 Corridors: Mean VV \u00b1 Std")

plt.tight_layout()
plt.savefig("../data/outputs/05_corridor_vv_statistics.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.5  Bi-temporal Change Detection

In [ ]:
# SAR DISCLAIMER: change detection results are SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

VV_BEFORE_PATH = Path(cfg["sources"]["sar"].get("sentinel1_vv_before", ""))
VV_AFTER_PATH  = Path(cfg["sources"]["sar"].get("sentinel1_vv_after", ""))

if VV_BEFORE_PATH.exists() and VV_AFTER_PATH.exists():
    print(f"Before scene: {VV_BEFORE_PATH}")
    print(f"After scene:  {VV_AFTER_PATH}")

    change_result = compute_backscatter_change(
        vv_before_path=VV_BEFORE_PATH,
        vv_after_path=VV_AFTER_PATH,
        corridor_buffer=corridors,
        thresholds_db=[3.0, 5.0, 8.0],
        before_date="2024-03-15",
        after_date="2024-04-18",
    )

    corridor_summary = change_result["corridor_change_summary"]
    print(f"Corridors with change summary: {len(corridor_summary):,}")

    if corridor_summary:
        summary_df = pd.DataFrame(corridor_summary)
        print("\nChange tier distribution:")
        print(summary_df["change_tier"].value_counts().to_string())
        flagged = summary_df[summary_df["flagged"]]
        print(f"\nFlagged corridors ({len(flagged):,}):")
        print(flagged[["corridor_id", "voltage_kv", "mean_ratio_db",
                        "max_abs_ratio_db", "change_tier"]].head(10).to_string(index=False))
else:
    print("SAR before/after scenes not found — skipping bi-temporal change analysis.")
    print(f"  Expected: {VV_BEFORE_PATH}")
    print(f"  Expected: {VV_AFTER_PATH}")
    print()
    print("To download Sentinel-1 scenes:")
    print("  from src.sar.sar_fetcher import SARFetcher")
    print("  fetcher = SARFetcher()")
    print("  fetcher.download_sentinel1(bbox=BBOX, date_range=('2024-03-01', '2024-04-30'))")

## 5.6  SAR Summary

In [ ]:
# SAR DISCLAIMER — required at end of every SAR notebook section
print(_SAR_DISCLAIMER)
print()

print("=" * 65)
print("  SAR CORRIDOR ANALYSIS SUMMARY")
print("=" * 65)
print(f"  source attribute:              SAR_contextual_RS")
print(f"  is_exact_asset_geometry:       False")
print(f"  Corridors extracted:           {len(corridors):,}")
print(f"  sar_confidence='unvalidated':  {n_unvalidated:,}")
print(f"  sar_confidence='uncertain':    {n_uncertain:,}  (spring-melt artefact)")
print()
print("  Buffer widths applied:")
for kv, m in buffer_widths_m.items():
    label = f"{kv} kV" if kv > 0 else "distribution"
    print(f"    {label:<16} {m} m")
print()
print("  Mean VV dB by voltage tier (synthetic where no raster):")
if "voltage_kv" in stats_df.columns and "mean_vv_db" in stats_df.columns:
    grp = stats_df.groupby("voltage_kv")["mean_vv_db"].mean().round(2)
    for kv, mean_db in grp.items():
        print(f"    {kv:.0f} kV  →  {mean_db:.2f} dB")
print("=" * 65)